<a href="https://colab.research.google.com/github/MarceCorreal2/Robots-NT/blob/main/Procesamiento_Indicadores_Backtest_V5ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Este cuaderno limpia y organiza los datos del resumen del strategy analyzer de los robots y debe incluir los indicadores en la
# tabla robots y procesar la calificación

## Procesamiento de Indicadores de Backtest

El objetivo de este cuaderno es tomar los datos crudos de los resúmenes del *strategy analyzer* de tus robots, limpiarlos y extraer los indicadores clave mencionados. Una vez procesados, los datos se guardarán en un formato limpio para futuros análisis.

In [129]:
# Celda 1 — Importes y configuración

import pandas as pd
import os
import re
from datetime import datetime
import pytz # Para manejar zonas horarias, movido aquí
import gspread # Para interactuar con Google Sheets
from google.colab import auth # Para la autenticación en Colab

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [130]:
pip install pytz

In [131]:
# Celda 2 — Conectar Drive

#from google.colab import drive
#drive.mount('/content/drive')

In [132]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [133]:
# Celda 2.1 - Autenticación con Google Drive y gspread (usando credenciales explícitas)

import google.auth # Importar google.auth

# Autenticar con Google para permitir el acceso a Google Sheets y Drive
# Usamos authenticate_user() para el flujo interactivo de Colab
auth.authenticate_user()

# Obtener las credenciales por defecto del usuario autenticado en Colab
creds, project = google.auth.default()

# Usar estas credenciales para inicializar el cliente de gspread
gc = gspread.Client(auth=creds)

print("Autenticación con Google Sheets completada usando credenciales de Google Colab.")

Autenticación con Google Sheets completada usando credenciales de Google Colab.


In [135]:
# Celda 13 - Cargar la Tabla de Calificación de Bots desde Google Sheets

# Nombre de tu Google Sheet (asegúrate de que coincida exactamente)
BOT_RATING_SHEET_NAME = 'Tabla de Calificación Bots'

try:
    # Abrir el Google Sheet por su nombre
    worksheet = gc.open(BOT_RATING_SHEET_NAME).sheet1

    # Obtener todos los datos como un DataFrame de pandas
    # `get_as_dataframe` lee la primera fila como encabezados automáticamente
    rating_table_df = pd.DataFrame(worksheet.get_all_records())

    print(f"Google Sheet '{BOT_RATING_SHEET_NAME}' cargado exitosamente.")
    print("Primeras 5 filas de la tabla de calificación:")
    display(rating_table_df.head())

    print("Columnas y tipos de datos de la tabla de calificación:")
    display(rating_table_df.info())

except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: El Google Sheet '{BOT_RATING_SHEET_NAME}' no se encontró.")
    print("Por favor, verifica que el nombre sea exacto y que tengas permisos de acceso.")
    rating_table_df = pd.DataFrame()
except Exception as e:
    print(f"Ocurrió un error al cargar el Google Sheet: {e}")
    rating_table_df = pd.DataFrame()

Google Sheet 'Tabla de Calificación Bots' cargado exitosamente.
Primeras 5 filas de la tabla de calificación:


,Indicador,Ponderación,Excelente,Aceptable,Malo
0,Profit Factor,20,>2.0,1.5–2.0,<1.5
1,Win Rate,10,>65%,55–65%,<55%
2,Recovery Factor,15,>3,2–3,<2
3,Sharpe Ratio,15,>1.5,1–1.5,<1
4,Drawdown Máximo,20,<10%,10–20%,>20%


Columnas y tipos de datos de la tabla de calificación:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Indicador    7 non-null      object
 1   Ponderación  7 non-null      int64 
 2   Excelente    7 non-null      object
 3   Aceptable    7 non-null      object
 4   Malo         7 non-null      object
dtypes: int64(1), object(4)
memory usage: 412.0+ bytes


None

In [136]:
# Celda 3 — Definir Variables

bot_name = 'RB001' # <-- Ajustado para que coincida con el nombre del archivo
test_number = 'T001' # <--- Corregido para que coincida con el archivo disponible
instrument = 'MNQ'

print(f"Bot Name: {bot_name}")
print(f"Test Number: {test_number}")
print(f"Instrument: {instrument}")

Bot Name: RB001
Test Number: T001
Instrument: MNQ


### Celda 3.1 — Añadir/Editar Comentarios para la Ejecución Actual

Utiliza el siguiente campo de texto para añadir o editar un comentario asociado a esta combinación de `Robot` y `Número de Test`. Si lo dejas en blanco, se mantendrá cualquier comentario existente para esta entrada en la tabla maestra. Si introduces texto, este sobrescribirá cualquier comentario previo.

In [137]:
#Celda 4 - Inicializa comentario

import ipywidgets as widgets
from IPython.display import display

# Initialize with an empty string or retrieve existing comment if possible
# For simplicity, we'll start with an empty string, assuming the user will input new comments.
# The logic in Celda 9 will handle retrieving existing comments if this input is left blank.
user_input_comment_widget = widgets.Textarea(
    value='',
    placeholder='Escribe tu comentario aquí para el robot y test actuales...',
    description='Comentario:',
    disabled=False,
    layout=widgets.Layout(width='auto', height='80px') # Adjust width to be flexible
)

def on_value_change(change):
    global comment_for_current_run
    comment_for_current_run = change.new

user_input_comment_widget.observe(on_value_change, names='value')

# Initialize global variable
comment_for_current_run = user_input_comment_widget.value

display(user_input_comment_widget)
print("Haz clic fuera del campo de texto o presiona Enter/Tab para guardar el comentario.")

Textarea(value='', description='Comentario:', layout=Layout(height='80px', width='auto'), placeholder='Escribe…

Haz clic fuera del campo de texto o presiona Enter/Tab para guardar el comentario.


In [138]:
# Celda 3.2 - Verificación del Comentario para la Ejecución Actual (opcional)

# Muestra el comentario que se usará para la ejecución actual, según lo ingresado en el widget.
# Esto te ayuda a confirmar que el valor está siendo capturado correctamente.
print(f"Comentario capturado para la ejecución actual: {comment_for_current_run if 'comment_for_current_run' in globals() else 'Ninguno (variable no definida o vacía)'}")

Comentario capturado para la ejecución actual: 


In [139]:
# Celda 4 — Definir Rutas dinámicas

RAW_DATA_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/'
CLEAN_DATA_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/'

print(f"Ruta de Datos Crudos: {RAW_DATA_PATH}")
print(f"Ruta de Datos Limpios: {CLEAN_DATA_PATH}")

Ruta de Datos Crudos: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/
Ruta de Datos Limpios: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/


In [140]:
#Celda 5 - Construir el nombre del archivo dinámicamente usando las variables definidas en Celda 3
# Ajustado para que el nombre del archivo sea `RB001_MNQ_A_T003.csv`

sample_file_name = f'SA_{bot_name}_{instrument}_A_{test_number}.csv' # Corrected filename to include '_A_'
sample_file_path = os.path.join(RAW_DATA_PATH, sample_file_name)

print(f"Cargando archivo de ejemplo: {sample_file_path}")

try:
    # Read the first few lines to understand the structure and find the actual data start
    raw_lines = []
    with open(sample_file_path, 'r', encoding='latin1') as f:
        for _ in range(30): # Read up to 30 lines to cover potential header length
            line = f.readline()
            if not line: # EOF
                break
            raw_lines.append(line.strip())

    print("\nPrimeras 20 líneas del archivo crudo para inspección:")
    for i, line in enumerate(raw_lines[:20]):
        print(f"Línea {i+1}: {line}")

    # Try to find the line that indicates the start of the actual performance metrics
    # Common indicators like 'Total net profit' usually appear at the start of data section.
    data_start_row = -1
    for i, line in enumerate(raw_lines):
        if 'Total net profit' in line:
            data_start_row = i
            break

    if data_start_row != -1:
        print(f"\nIdentificado el inicio de los datos de indicadores en la línea (0-index): {data_start_row}")
        # Read the CSV again, skipping lines up to the identified data start
        # We set header=None because the first column will contain the indicator names,
        # and the subsequent columns are values (e.g., All trades, Long trades, Short trades).
        sample_df = pd.read_csv(sample_file_path, encoding='latin1', sep=';', skiprows=data_start_row, header=None)

        # Assuming the first column is the indicator name and the next are its values
        # We need to clean up the column names based on the context. Sample: Performance;All trades;Long trades;Short trades
        # Let's just display the raw parsed DataFrame for now.
        print("\nDataFrame de indicadores procesado (primeras 5 filas):")
        print(sample_df.head().to_markdown(index=False, numalign="left", stralign="left"))
        print("\nColumnas del DataFrame procesado:")
        print(sample_df.columns.tolist())
    else:
        print("\nNo se pudo identificar el inicio de los datos de indicadores ('Total net profit' no encontrado). Se muestra la lectura inicial sin procesar.")
        # Fallback if specific data start not found, try reading with just separator
        sample_df = pd.read_csv(sample_file_path, encoding='latin1', sep=';')
        print("\nPrimeras 5 filas del archivo de ejemplo (lectura básica):")
        print(sample_df.head().to_markdown(index=False, numalign="left", stralign="left"))
        print("\nColumnas del archivo de ejemplo (lectura básica):")
        print(sample_df.columns.tolist())

except FileNotFoundError:
    print(f"Error: El archivo '{sample_file_name}' no se encontró en la ruta '{RAW_DATA_PATH}'. Por favor, verifica la ruta y el nombre del archivo.")
    print("\nArchivos CSV disponibles en la carpeta de datos crudos:")
    csv_files = [f for f in os.listdir(RAW_DATA_PATH) if f.endswith('.csv')]
    if csv_files:
        for f in csv_files:
            print(f"- {f}")
    else:
        print("No se encontraron archivos CSV en esta ruta.")
except Exception as e:
    print(f"Error al leer el archivo CSV: {e}")

Cargando archivo de ejemplo: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/SA_RB001_MNQ_A_T001.csv

Primeras 20 líneas del archivo crudo para inspección:
Línea 1: Performance;All trades;Long trades;Short trades;
Línea 2: Total net profit;$ 5680,50;$ 5680,50;$ 0,00;
Línea 3: Gross profit;$ 30262,50;$ 30262,50;$ 0,00;
Línea 4: Gross loss;-$ 24582,00;-$ 24582,00;$ 0,00;
Línea 5: Commission;$ 1045,00;$ 1045,00;$ 0,00;
Línea 6: Profit factor;1,23;1,23;1,00;
Línea 7: Max drawdown;-$ 6111,80;-$ 6111,80;$ 0,00;
Línea 8: Sharpe ratio;0,26;0,26;1,00;
Línea 9: Sortino ratio;0,60;0,60;1,00;
Línea 10: Ulcer index;0,05;0,05;0,00;
Línea 11: R squared;0,39;0,39;0,00;
Línea 12: Total Fees;$ 0,00;$ 0,00;$ 0,00;
Línea 13: Probability;12,52 %;12,52 %;0,00 %;
Línea 14: ;;;;
Línea 15: Start date;1/01/2026;;;
Línea 16: Start time;12:00 AM;;;
Línea 17: End date;30/06/2026;;;
Línea 18: End time;12:00 AM;;;
Línea 19: ;;;;
Línea 20: Total # of trades;550;550;0;

Identificado el inici

In [141]:
import pandas as pd
import os
import re
from datetime import datetime
import pytz # Para manejar zonas horarias, movido aquí
import gspread # Para interactuar con Google Sheets
from google.colab import auth # Para la autenticación en Colab

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Celda 6 - Limpieza de datos

# Lista para almacenar los DataFrames de indicadores de cada archivo
all_indicators_list = []

# --- AÑADIDO: Asegurar que sample_file_name sea el correcto para esta celda ---
sample_file_name = f'SA_{bot_name}_{instrument}_A_{test_number}.csv' # Corrected filename to include '_A_'
# --- FIN AÑADIDO ---

# Función para limpiar y convertir valores numéricos
def clean_numeric_value(value):
    if isinstance(value, str):
        # CORRECCIÓN: Usar '.replace(' ', '')' en lugar de 're.escape(' ')'
        value = value.replace('$', '').replace(' ', '').replace('%', '').replace(',', '.')
        if value == '' or value == '-':
            return None
        try:
            return float(value)
        except ValueError:
            return value
    return value

# Usar el sample_file_name y sample_file_path ya definidos en Celda 5
# para procesar solo el archivo deseado.
print(f"Procesando el archivo especificado: {sample_file_name}")

# full_file_path ya está definido en Celda 5 y es el que queremos procesar
# Construimos full_file_path nuevamente aquí para asegurar que sea el correcto
# si Celda 5 no se ejecuta justo antes.
full_file_path = os.path.join(RAW_DATA_PATH, sample_file_name)

try:
    # --- Parsing robusto (similar a Celda 5) ---
    raw_lines = []
    with open(full_file_path, 'r', encoding='latin1') as f:
        for _ in range(30): # Read up to 30 lines to find data start
            line = f.readline()
            if not line: break
            raw_lines.append(line.strip())

    data_start_row = -1
    for i, line in enumerate(raw_lines):
        if 'Total net profit' in line:
            data_start_row = i
            break

    if data_start_row == -1:
        print(f"Advertencia: No se encontró el inicio de datos para {sample_file_name}. No se procesará este archivo.")
    else:
        temp_df = pd.read_csv(full_file_path, encoding='latin1', sep=';', skiprows=data_start_row, header=None)

        # --- Limpieza y extracción (similar a Celda 6)---
        # Aplicar nombres de columna iniciales y establecer índice
        temp_df.columns = ['Performance', 'All trades', 'Long trades', 'Short trades', 'Extra_Column']
        temp_df = temp_df.drop(columns=['Extra_Column'])
        temp_df['Performance'] = temp_df['Performance'].str.strip()
        temp_df = temp_df.set_index('Performance')

        # Aplicar limpieza a valores numéricos
        for col in ['All trades', 'Long trades', 'Short trades']:
            temp_df[col] = temp_df[col].apply(clean_numeric_value)

        def get_indicator_value(df, indicator_name):
            try:
                return df.loc[indicator_name.strip(), 'All trades']
            except KeyError:
                return None

        # Extraer los indicadores solicitados
        TotalNetProfit = get_indicator_value(temp_df, 'Total net profit')
        PF = get_indicator_value(temp_df, 'Profit factor')
        WR_probability = get_indicator_value(temp_df, 'Probability')
        MaxDrawdown = get_indicator_value(temp_df, 'Max drawdown') # Ahora debería ser float
        TotalTrades = get_indicator_value(temp_df, 'Total # of trades')
        Winners = get_indicator_value(temp_df, 'Number of winning trades')
        GrossProfit = get_indicator_value(temp_df, 'Gross profit')
        GrossLoss = get_indicator_value(temp_df, 'Gross loss')
        AvgWinTrade = get_indicator_value(temp_df, 'Avg winning trade') # Ahora debería ser float
        AvgLossTrade = get_indicator_value(temp_df, 'Avg losing trade') # Ahora debería ser float

        WR = WR_probability if WR_probability is not None else \
             (Winners / TotalTrades) * 100 if TotalTrades and Winners is not None and TotalTrades != 0 else None

        # Corrected PayoffRatio calculation
        PayoffRatio = (AvgWinTrade / abs(AvgLossTrade)) if AvgWinTrade is not None and AvgLossTrade is not None and AvgLossTrade != 0 else None

        # Calculate RecoveryFactor as Total Net Profit / abs(Max Drawdown)
        RecoveryFactor = (TotalNetProfit / abs(MaxDrawdown)) if TotalNetProfit is not None and MaxDrawdown is not None and MaxDrawdown != 0 else None

        # Extracción de fechas y cálculo de Net Profit/Mes
        FechaInicio = None
        FechaFin = None
        for line in raw_lines:
            if 'Start date' in line:
                match = re.search(r'Start date;(\d{1,2}/\d{1,2}/\d{4});', line)
                if match:
                    FechaInicio = datetime.strptime(match.group(1), '%d/%m/%Y')
            elif 'End date' in line:
                match = re.search(r'End date;(\d{1,2}/\d{1,2}/\d{4});', line)
                if match:
                    FechaFin = datetime.strptime(match.group(1), '%d/%m/%Y')

        NumMonths = None
        NetProfitPerMonth = None
        if TotalNetProfit is not None and FechaInicio is not None and FechaFin is not None:
            delta = FechaFin - FechaInicio
            if delta.days > 0:
                NumMonths = delta.days / 30.44
                if NumMonths > 0:
                    NetProfitPerMonth = TotalNetProfit / NumMonths

        # Debug print: Verificar los valores antes de crear el diccionario
        print(f"\nValores a usar para Robot Base: {bot_name}, Test ID: {test_number}, Instrumento: {instrument}")

        # Crear un diccionario con los indicadores para este archivo
        file_indicators = {
            'Archivo': sample_file_name,
            'Robot Base': bot_name,
            'Test ID': test_number,
            'Instrumento': instrument,
            'Total net profit': TotalNetProfit,
            'PF': PF,
            'Probability': WR, # CAMBIO: 'WR' renombrado a 'Probability' para los archivos limpios
            'Max drawdown': MaxDrawdown, # Actualizado a 'Max drawdown'
            'Recovery Factor': RecoveryFactor,
            'PayoffRatio': PayoffRatio,
            'Total # of trades': TotalTrades,
            '# Meses': NumMonths,
            'Net Profit/Mes': NetProfitPerMonth,
            'Fecha-Inicio': FechaInicio.strftime('%Y-%m-%d') if FechaInicio else None,
            'Fecha-Fin': FechaFin.strftime('%Y-%m-%d') if FechaFin else None,
            'Avg Win': AvgWinTrade,
            'Avg Loss': AvgLossTrade
        }
        all_indicators_list.append(file_indicators)

        # --- Nuevo código para guardar el archivo limpio individual sin subdirectorios dinámicos ---
        individual_df = pd.DataFrame([file_indicators])
        base_file_name = os.path.splitext(sample_file_name)[0]
        cleaned_individual_file_name = f"{base_file_name}_Limpio.csv"

        dynamic_output_dir = CLEAN_DATA_PATH
        os.makedirs(dynamic_output_dir, exist_ok=True)

        individual_output_path = os.path.join(dynamic_output_dir, cleaned_individual_file_name)
        individual_df.to_csv(individual_output_path, index=False)
        print(f"Indicadores individuales guardados en: {individual_output_path}")
        # --- Fin del nuevo código ---

except FileNotFoundError:
    print(f"Error: El archivo '{sample_file_name}' no se encontró en la ruta '{RAW_DATA_PATH}'. Por favor, verifica la ruta y el nombre del archivo.")
except Exception as e:
    print(f"Error al procesar el archivo {sample_file_name}: {e}")

# Convertir la lista de diccionarios a un DataFrame consolidado
if all_indicators_list:
    consolidated_df = pd.DataFrame(all_indicators_list)
    print("\n--- DataFrame Consolidado de Indicadores (primeras 5 filas): ---")
    display(consolidated_df.head())

    # No guardar el DataFrame consolidado aquí ya que la Celda 9 se encarga de la tabla maestra.
elif 'consolidated_df' not in globals(): # Solo imprimir si no se pudo crear consolidated_df
    print("No se pudieron procesar indicadores de ningún archivo.")

Procesando el archivo especificado: SA_RB001_MNQ_A_T001.csv

Valores a usar para Robot Base: RB001, Test ID: T001, Instrumento: MNQ
Indicadores individuales guardados en: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/SA_RB001_MNQ_A_T001_Limpio.csv

--- DataFrame Consolidado de Indicadores (primeras 5 filas): ---


,Archivo,Robot Base,Test ID,Instrumento,Total net profit,PF,Probability,Max drawdown,Recovery Factor,PayoffRatio,Total # of trades,# Meses,Net Profit/Mes,Fecha-Inicio,Fecha-Fin,Avg Win,Avg Loss
0,SA_RB001_MNQ_A_T001.csv,RB001,T001,MNQ,5680.5,1.23,12.52,-6111.8,0.929432,7.232887,550.0,5.913272,960.635667,2026-01-01,2026-06-30,378.28,-52.3


In [142]:
# Celda 7 - Limpieza y reordenamiento de datos finales


# Check if consolidated_df exists. If not, try to reconstruct it from the last saved individual cleaned file.
if 'consolidated_df' not in locals() and 'consolidated_df' not in globals():
    print("Advertencia: 'consolidated_df' no definido en el estado actual del kernel. Intentando cargar el último archivo limpio individual.")
    try:
        # Assuming bot_name, test_number, and CLEAN_DATA_PATH are defined in previous cells and are accessible.
        # Corregido: Usar el formato de nombre de archivo consistente con Celda 5.
        cleaned_individual_file_name = f"{bot_name}{test_number}-Limpio.csv"
        individual_output_path = os.path.join(CLEAN_DATA_PATH, cleaned_individual_file_name)

        if os.path.exists(individual_output_path):
            consolidated_df = pd.read_csv(individual_output_path)
            print(f"Éxito: 'consolidated_df' cargado desde {individual_output_path}.")
        else:
            print(f"Error: No se pudo cargar 'consolidated_df'. El archivo '{individual_output_path}' no existe. Por favor, asegúrese de ejecutar la 'Celda 6' primero.")
            consolidated_df = pd.DataFrame() # Create an empty DataFrame to prevent subsequent errors
    except NameError as e:
        print(f"Error de variable al intentar cargar 'consolidated_df': {e}. Asegúrese de que 'bot_name', 'test_number' y 'CLEAN_DATA_PATH' estén definidos en celdas anteriores.")
        consolidated_df = pd.DataFrame() # Create an empty DataFrame as a last resort
    except Exception as e:
        print(f"Error inesperado al cargar 'consolidated_df': {e}")
        consolidated_df = pd.DataFrame() # Create an empty DataFrame as a last resort


# Proceed only if consolidated_df is not empty after the potential loading attempt
if not consolidated_df.empty:
    # Renombrar columnas para que coincidan exactamente con la solicitud del usuario
    # Make a copy to avoid SettingWithCopyWarning later, especially during column renames
    consolidated_df = consolidated_df.copy().rename(columns={
        'Fecha-Inicio': 'Fecha Inicio',
        'Fecha-Fin': 'Fecha Fin',
        'Max drawdown': 'Max drawdown', # Updated to 'Max drawdown'
        'Net Profit/Mes': 'Net Profit/Mes',
        'Total net profit': 'Total net profit'
    })

    # Definir el orden de las columnas solicitado por el usuario, incluyendo 'Robot Base' y 'Test ID'
    column_order = [
        'Fecha Inicio',
        'Fecha Fin',
        'Instrumento',
        'Robot Base',
        'Test ID',
        '# Meses',
        'Total # of trades',
        'Total net profit',
        'Net Profit/Mes',
        'PF',
        'Probability', # CAMBIO: Esperar 'Probability' en lugar de 'WR'
        'Max drawdown', # Updated to 'Max drawdown'
        'Recovery Factor',
        'PayoffRatio',
        'Avg Win',
        'Avg Loss'
    ]

    # Seleccionar y reordenar las columnas del DataFrame
    # Filter column_order to only include columns actually present in consolidated_df
    actual_columns_in_order = [col for col in column_order if col in consolidated_df.columns]
    final_df = consolidated_df[actual_columns_in_order]

    print("\n--- DataFrame Final con Columnas Limpias y Reordenadas (primeras 5 filas): ---")
    display(final_df.head())
else:
    final_df = pd.DataFrame() # Ensure final_df is defined even if consolidated_df is empty
    print("No se pudo generar 'final_df' porque 'consolidated_df' está vacío o no se pudo cargar.")


--- DataFrame Final con Columnas Limpias y Reordenadas (primeras 5 filas): ---


,Fecha Inicio,Fecha Fin,Instrumento,Robot Base,Test ID,# Meses,Total # of trades,Total net profit,Net Profit/Mes,PF,Probability,Max drawdown,Recovery Factor,PayoffRatio,Avg Win,Avg Loss
0,2026-01-01,2026-06-30,MNQ,RB001,T001,5.913272,550.0,5680.5,960.635667,1.23,12.52,-6111.8,0.929432,7.232887,378.28,-52.3


In [143]:
# Celda 8 - Verificar tipos de datos del DataFrame final
print("\n--- Tipos de datos del DataFrame final: ---")
display(final_df.dtypes)


--- Tipos de datos del DataFrame final: ---


,0
Fecha Inicio,object
Fecha Fin,object
Instrumento,object
Robot Base,object
Test ID,object
# Meses,float64
Total # of trades,float64
Total net profit,float64
Net Profit/Mes,float64
PF,float64


In [144]:
import pandas as pd
import os
import re
from datetime import datetime
import pytz # Importar pytz

# Define the base path for the master table
BASE_MASTER_TABLE_DIR = '/content/drive/MyDrive/Robots/Mis Bots/'
BASE_MASTER_TABLE_NAME = 'Tabla Maestra Bots'

# Helper function to extract Robot and Test ID from a string, designed to handle various formats
def parse_robot_test_id(s):
    if pd.isna(s):
        return None, None
    s = str(s).strip()

    robot_extracted = None
    test_id_extracted = None

    # Try to extract Robot (e.g., RB001)
    match_robot = re.search(r'RB(\d+)', s, re.IGNORECASE)
    if match_robot:
        robot_extracted = f"RB{match_robot.group(1).zfill(3)}" # Ensure 3 digits, e.g., RB001

    # Try to extract Test ID (e.g., T001)
    match_test_id = re.search(r'T(\d+)', s, re.IGNORECASE)
    if match_test_id:
        test_id_extracted = f"T{match_test_id.group(1).zfill(3)}" # Ensure 3 digits, e.g., T001

    return robot_extracted, test_id_extracted


# Renombrar columnas en final_df antes de cualquier otra operación para asegurar consistencia
# Usamos .copy() para evitar SettingWithCopyWarning
final_df_to_add = final_df.rename(columns={'Robot Base': 'Robot', 'Test ID': 'Numero del Test', 'WR': 'Probability'}).copy()

# Añadir la columna 'Comentarios' con valores vacíos al DataFrame que se va a añadir
# Esto asegura que la columna exista y no sea sobrescrita por lógica del notebook.
# Este valor será sobrescrito si existe un comentario anterior para este robot/test.
final_df_to_add['Comentarios'] = ''

# Añadir la columna 'Calificación' con valores nulos iniciales
final_df_to_add['Calificación'] = pd.NA


# Asegurar que las columnas clave sean de tipo string para la comparación
final_df_to_add['Robot'] = final_df_to_add['Robot'].astype(str)
final_df_to_add['Numero del Test'] = final_df_to_add['Numero del Test'].astype(str)

# Convertir las columnas de fecha en final_df_to_add a tipo datetime para compatibilidad
for col in ['Fecha Inicio', 'Fecha Fin']:
    if col in final_df_to_add.columns:
        final_df_to_add.loc[:, col] = pd.to_datetime(final_df_to_add[col], errors='coerce')


# Construct the path to the most recent master table for loading
# We'll look for any CSV starting with BASE_MASTER_TABLE_NAME
master_files = [f for f in os.listdir(BASE_MASTER_TABLE_DIR) if f.startswith(BASE_MASTER_TABLE_NAME) and f.endswith('.csv')]
master_files.sort(reverse=True) # Sort to get the most recent file first

current_master_table_path = None
if master_files:
    current_master_table_path = os.path.join(BASE_MASTER_TABLE_DIR, master_files[0])


# Verificar si el archivo de la tabla maestra existe
if current_master_table_path and os.path.exists(current_master_table_path):
    print(f"Cargando tabla maestra más reciente desde: {current_master_table_path}")
    master_df = pd.read_csv(current_master_table_path)

    # IMPORTE: Renombrar columnas antiguas en master_df para que coincidan con los nuevos nombres estandarizados
    master_df = master_df.rename(columns={
        '# Trades': 'Total # of trades',
        'NetProfit': 'Total net profit',
        'DD Max': 'Max drawdown',
        'WR': 'Probability'
    })

    # --- START: Robust standardization of 'Robot' and 'Numero del Test' in loaded master_df ---
    # First, ensure 'Robot' and 'Numero del Test' columns exist, possibly from old names
    if 'Robot Base' in master_df.columns and 'Robot' not in master_df.columns:
        master_df = master_df.rename(columns={'Robot Base': 'Robot'})
    if 'Test ID' in master_df.columns and 'Numero del Test' not in master_df.columns:
        master_df = master_df.rename(columns={'Test ID': 'Numero del Test'})

    # If the columns still don't exist, create them with placeholder
    if 'Robot' not in master_df.columns:
        master_df['Robot'] = pd.NA
    if 'Numero del Test' not in master_df.columns:
        master_df['Numero del Test'] = pd.NA

    # Apply robust parsing to standardize Robot and Numero del Test
    standardized_robot_col = []
    standardized_test_id_col = []

    for idx, row in master_df.iterrows():
        # Get current values, try to use existing if they seem valid
        current_robot_val = row['Robot']
        current_test_id_val = row['Numero del Test']

        # Attempt to parse from current 'Robot' value
        parsed_r_from_robot, parsed_t_from_robot = parse_robot_test_id(current_robot_val)
        # Attempt to parse from current 'Numero del Test' value
        parsed_r_from_test_id, parsed_t_from_test_id = parse_robot_test_id(current_test_id_val)
        # Attempt to parse from 'Archivo' column if it exists and current keys are problematic
        parsed_r_from_file = None
        parsed_t_from_file = None
        if 'Archivo' in master_df.columns and (pd.isna(current_robot_val) or pd.isna(current_test_id_val) or not str(current_robot_val).startswith('RB') or not str(current_test_id_val).startswith('T')):
             parsed_r_from_file, parsed_t_from_file = parse_robot_test_id(row['Archivo'])


        # Prioritize values that look correct (e.g., start with 'RB'/'T')
        # Combine parsed results, prioritizing from specific columns or more complete sources
        final_robot = None
        if parsed_r_from_robot and parsed_r_from_robot.startswith('RB'):
            final_robot = parsed_r_from_robot
        elif parsed_r_from_test_id and parsed_r_from_test_id.startswith('RB'):
            final_robot = parsed_r_from_test_id
        elif parsed_r_from_file and parsed_r_from_file.startswith('RB'):
            final_robot = parsed_r_from_file

        final_test_id = None
        if parsed_t_from_test_id and parsed_t_from_test_id.startswith('T'):
            final_test_id = parsed_t_from_test_id
        elif parsed_t_from_robot and parsed_t_from_robot.startswith('T'):
            final_test_id = parsed_t_from_robot
        elif parsed_t_from_file and parsed_t_from_file.startswith('T'):
            final_test_id = parsed_t_from_file

        standardized_robot_col.append(final_robot if final_robot else 'UNKNOWN_ROBOT')
        standardized_test_id_col.append(final_test_id if final_test_id else 'UNKNOWN_TEST')

    master_df['Robot'] = standardized_robot_col
    master_df['Numero del Test'] = standardized_test_id_col

    # Ensure 'Robot' and 'Numero del Test' are string type after standardization
    master_df['Robot'] = master_df['Robot'].astype(str)
    master_df['Numero del Test'] = master_df['Numero del Test'].astype(str)
    # --- END: Robust standardization of 'Robot' and 'Numero del Test' in loaded master_df ---

    # --- ADDED: Clean up potentially erroneous columns from loaded master_df BEFORE merging ---
    # Drop 'NetProfit/Mes' (without space) if 'Net Profit/Mes' (with space) exists, as the latter is correct.
    if 'NetProfit/Mes' in master_df.columns and 'Net Profit/Mes' in master_df.columns:
        print("Detectado y eliminando columna 'NetProfit/Mes' (sin espacio) de la tabla maestra cargada.")
        master_df = master_df.drop(columns=['NetProfit/Mes'])
    # Rename 'NetProfit/Mes' (without space) to 'Net Profit/Mes' (with space) if only the former exists.
    elif 'NetProfit/Mes' in master_df.columns and 'Net Profit/Mes' not in master_df.columns:
        print("Renombrando columna 'NetProfit/Mes' a 'Net Profit/Mes' en la tabla maestra cargada.")
        master_df = master_df.rename(columns={'NetProfit/Mes': 'Net Profit/Mes'})

    # Drop 'Calificación' if it exists in the loaded master_df, as it will be recalculated later.
    if 'Calificación' in master_df.columns:
        print("Eliminando columna 'Calificación' de la tabla maestra cargada para recalcularla.")
        master_df = master_df.drop(columns=['Calificación'])

    # Add 'Comentarios' column to master_df if it doesn't exist, initialized with empty strings
    if 'Comentarios' not in master_df.columns:
        master_df['Comentarios'] = ''

    # --- MODIFICADO: Recuperar y COMBINAR el comentario existente ---
    combined_comment_for_current_run = ''
    if not final_df_to_add.empty:
        current_robot_id = final_df_to_add['Robot'].iloc[0]
        current_test_id = final_df_to_add['Numero del Test'].iloc[0]

        # Find if this specific (robot, test) pair exists in the loaded master_df
        matching_rows = master_df[(master_df['Robot'] == current_robot_id) & (master_df['Numero del Test'] == current_test_id)]

        existing_comment = ''
        if not matching_rows.empty and 'Comentarios' in matching_rows.columns:
            retrieved_comment = matching_rows['Comentarios'].iloc[0]
            if pd.notna(retrieved_comment):
                existing_comment = str(retrieved_comment).strip()

        # Get the new comment from the widget (defined in the previous step)
        # Ensure comment_for_current_run is accessible and defaults to empty string if not set
        new_comment_from_widget = globals().get('comment_for_current_run', '').strip()

        if existing_comment and new_comment_from_widget:
            combined_comment_for_current_run = f"{existing_comment}. {new_comment_from_widget}"
        elif existing_comment:
            combined_comment_for_current_run = existing_comment
        elif new_comment_from_widget:
            combined_comment_for_current_run = new_comment_from_widget

    # Asignar el comentario combinado a la fila que se va a añadir
    if not final_df_to_add.empty:
        final_df_to_add.loc[:, 'Comentarios'] = combined_comment_for_current_run
    # --- FIN MODIFICACIÓN DE COMBINACIÓN DE COMENTARIOS ---

    # --- END ADDED CLEANUP ---

    # Convertir las columnas de fecha en master_df a tipo datetime si es necesario
    for col in ['Fecha Inicio', 'Fecha Fin']:
        if col in master_df.columns:
            # Convert to string first to avoid errors with mixed types, then to datetime
            master_df[col] = master_df[col].astype(str)
            master_df.loc[:, col] = pd.to_datetime(master_df[col], errors='coerce')


    print("\n--- Tabla Maestra Original (primeras 5 filas estandarizadas y limpiadas): ---")
    display(master_df.head())

    # Get all unique (Robot, Numero del Test) pairs from the current processed data
    processed_keys = final_df_to_add[['Robot', 'Numero del Test']].drop_duplicates()

    # Create a boolean mask to identify rows in master_df that should be removed
    # These are rows whose (Robot, Numero del Test) pair is present in processed_keys
    mask_to_remove = master_df.set_index(['Robot', 'Numero del Test']).index.isin(
        processed_keys.set_index(['Robot', 'Numero del Test']).index
    )

    # Filter master_df to remove rows that are present in the current processed data
    master_df_filtered = master_df[~mask_to_remove].copy()

    # Concatenate the filtered master_df with the new records
    updated_master_df = pd.concat([master_df_filtered, final_df_to_add], ignore_index=True)

else:
    print(f"Advertencia: No se encontró una tabla maestra existente. Creando una nueva tabla maestra con los datos actuales.")
    updated_master_df = final_df_to_add.copy() # La primera entrada será la ejecución actual

# --- NUEVA LÓGICA DE LIMPIEZA DE COLUMNAS SIMILARES (retained for final check) ---
# This block acts as a fallback to ensure that after concatenation,
# if 'NetProfit/Mes' (without space) is still present and 'Net Profit/Mes' (with space) also exists,
# the former is removed. This might catch issues if final_df_to_add somehow introduced it, though unlikely.
if 'NetProfit/Mes' in updated_master_df.columns and 'Net Profit/Mes' in updated_master_df.columns:
    print("Detectado y eliminando columna 'NetProfit/Mes' (sin espacio) duplicada en el DataFrame maestro actualizado.")
    updated_master_df = updated_master_df.drop(columns=['NetProfit/Mes'])
elif 'NetProfit/Mes' in updated_master_df.columns and 'Net Profit/Mes' not in updated_master_df.columns:
    print("Renombrando columna 'NetProfit/Mes' a 'Net Profit/Mes' en el DataFrame maestro actualizado.")
    updated_master_df = updated_master_df.rename(columns={'NetProfit/Mes': 'Net Profit/Mes'})
# --- FIN NUEVA LÓGICA DE LIMPIEZA ---


# Reordenar columnas para colocar 'Robot' y 'Numero del Test' al principio
expected_cols_order = [
    'Robot', 'Numero del Test', 'Fecha Inicio', 'Fecha Fin', 'Instrumento',
    '# Meses', 'Total # of trades', 'Total net profit', 'Net Profit/Mes', 'PF', 'Probability',
    'Max drawdown', # CORRECCIÓN: Actualizado a 'Max drawdown'
    'Recovery Factor', 'PayoffRatio', 'Avg Win', 'Avg Loss',
    'Calificación',
    'Comentarios'
]
# Add any missing columns to updated_master_df that are in expected_cols_order, filling with NaN
for col in expected_cols_order:
    if col not in updated_master_df.columns:
        updated_master_df[col] = pd.NA

# Reorder columns based on expected_cols_order
updated_master_df = updated_master_df[expected_cols_order]

# Sort the DataFrame by 'Robot' and 'Numero del Test' as requested by the user
updated_master_df = updated_master_df.sort_values(by=['Robot', 'Numero del Test']).reset_index(drop=True)


print("\n--- Tabla Maestra Actualizada (primeras 5 filas incluyendo los nuevos datos): ---")
display(updated_master_df.head())

# Define the Miami timezone
miami_timezone = pytz.timezone('America/New_York')

# Guardar la tabla maestra actualizada con la fecha y hora actual en la zona horaria de Miami
current_date_str = datetime.now(miami_timezone).strftime('%Y-%m-%d_%H-%M-%S')
MASTER_TABLE_PATH_DATED = os.path.join(BASE_MASTER_TABLE_DIR, f"{BASE_MASTER_TABLE_NAME}_{current_date_str}.csv")

# --- NUEVO: Eliminar archivos de tabla maestra anteriores ---
print("\nEliminando versiones anteriores de la tabla maestra...")
for f_name in os.listdir(BASE_MASTER_TABLE_DIR):
    if f_name.startswith(BASE_MASTER_TABLE_NAME) and f_name.endswith('.csv'):
        file_path_to_delete = os.path.join(BASE_MASTER_TABLE_DIR, f_name)
        try:
            os.remove(file_path_to_delete)
            print(f"  - Eliminado: {f_name}")
        except Exception as e:
            print(f"  - Error al eliminar {f_name}: {e}")
# --- FIN NUEVO ---

updated_master_df.to_csv(MASTER_TABLE_PATH_DATED, index=False)
print(f"\nTabla maestra actualizada y guardada en: {MASTER_TABLE_PATH_DATED}")

Advertencia: No se encontró una tabla maestra existente. Creando una nueva tabla maestra con los datos actuales.

--- Tabla Maestra Actualizada (primeras 5 filas incluyendo los nuevos datos): ---


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,Total # of trades,Total net profit,Net Profit/Mes,PF,Probability,Max drawdown,Recovery Factor,PayoffRatio,Avg Win,Avg Loss,Calificación,Comentarios
0,RB001,T001,2026-01-01 00:00:00,2026-06-30 00:00:00,MNQ,5.913272,550.0,5680.5,960.635667,1.23,12.52,-6111.8,0.929432,7.232887,378.28,-52.3,<NA>,



Eliminando versiones anteriores de la tabla maestra...

Tabla maestra actualizada y guardada en: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots_2026-08-20_16-31-52.csv


In [145]:
# Celda 10 - Verificar datos

# Define the base path for the master table (same as in Celda 9)
BASE_MASTER_TABLE_DIR = '/content/drive/MyDrive/Robots/Mis Bots/'
BASE_MASTER_TABLE_NAME = 'Tabla Maestra Bots'

# *** MODIFICACIÓN: Usar directamente la ruta del archivo recién guardado por Celda 9 ***
# Assuming MASTER_TABLE_PATH_DATED was set by the last run of Celda 9
# If Celda 9 hasn't run yet, or if the variable is not available for some reason,
# we'll fall back to searching for the latest file, but prioritize the explicit path.

MASTER_TABLE_PATH = None
if 'MASTER_TABLE_PATH_DATED' in globals():
    MASTER_TABLE_PATH = globals()['MASTER_TABLE_PATH_DATED']
else:
    # Fallback: Get all master table files and sort them to find the most recent one
    master_files = [f for f in os.listdir(BASE_MASTER_TABLE_DIR) if f.startswith(BASE_MASTER_TABLE_NAME) and f.endswith('.csv')]
    master_files.sort(reverse=True) # Sort to get the most recent file first
    if master_files:
        MASTER_TABLE_PATH = os.path.join(BASE_MASTER_TABLE_DIR, master_files[0])


if MASTER_TABLE_PATH:
    print(f"Verificando el contenido de la tabla maestra más reciente: {MASTER_TABLE_PATH}")
    if os.path.exists(MASTER_TABLE_PATH):
        verified_master_df = pd.read_csv(MASTER_TABLE_PATH)

        # Mostrar el DataFrame actualizado con todos los indicadores
        print("Tabla Maestra Actualizada con todos los Indicadores y Calificaciones:")
        display(verified_master_df)

        print(f"Total de filas en la tabla maestra: {len(verified_master_df)}")
    else:
        print(f"Error: El archivo '{MASTER_TABLE_PATH}' no existe o no se pudo cargar.")
else:
    print(f"El archivo de la Tabla Maestra (con prefijo '{BASE_MASTER_TABLE_NAME}') no se encontró en '{BASE_MASTER_TABLE_DIR}'.")


Verificando el contenido de la tabla maestra más reciente: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots_2026-08-20_16-31-52.csv
Tabla Maestra Actualizada con todos los Indicadores y Calificaciones:


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,Total # of trades,Total net profit,Net Profit/Mes,PF,Probability,Max drawdown,Recovery Factor,PayoffRatio,Avg Win,Avg Loss,Calificación,Comentarios
0,RB001,T001,2026-01-01 00:00:00,2026-06-30 00:00:00,MNQ,5.913272,550.0,5680.5,960.635667,1.23,12.52,-6111.8,0.929432,7.232887,378.28,-52.3,NaN,NaN


Total de filas en la tabla maestra: 1


### Celdas Deprecadas: `gspread` y Autenticación (Celda 11 y Celda 12)

Las Celdas 11 (instalación de `gspread`) y 12 (autenticación de `gspread` con `service_account()`) ya no son necesarias.

*   **Instalación:** `gspread` ya se instala y/o se importa al inicio del notebook en la Celda 1.
*   **Autenticación:** La autenticación correcta para `gspread` en Google Colab se realiza en la **Celda 2.1** utilizando las credenciales de usuario de Colab (`google.auth`). La 'Celda 12' intentaba usar `gspread.service_account()`, lo cual requiere un archivo de credenciales de cuenta de servicio que no está presente en este flujo.

Por favor, **ignora o elimina manualmente estas celdas (Celda 11 y Celda 12)** para evitar errores y mantener el notebook limpio, ya que su funcionalidad ha sido reemplazada y mejorada en otras secciones.

In [146]:
# Celda 13 - Cargar la Tabla de Calificación de Bots desde Google Sheets

# Nombre de tu Google Sheet (asegúrate de que coincida exactamente)
BOT_RATING_SHEET_NAME = 'Tabla de Calificación Bots'

try:
    # Abrir el Google Sheet por su nombre
    worksheet = gc.open(BOT_RATING_SHEET_NAME).sheet1

    # Obtener todos los datos como un DataFrame de pandas
    # `get_as_dataframe` lee la primera fila como encabezados automáticamente
    rating_table_df = pd.DataFrame(worksheet.get_all_records())

    print(f"Google Sheet '{BOT_RATING_SHEET_NAME}' cargado exitosamente.")
    print("Primeras 5 filas de la tabla de calificación:")
    display(rating_table_df.head())

    print("Columnas y tipos de datos de la tabla de calificación:")
    display(rating_table_df.info())

except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: El Google Sheet '{BOT_RATING_SHEET_NAME}' no se encontró.")
    print("Por favor, verifica que el nombre sea exacto y que tengas permisos de acceso.")
    rating_table_df = pd.DataFrame()
except Exception as e:
    print(f"Ocurrió un error al cargar el Google Sheet: {e}")
    rating_table_df = pd.DataFrame()

Google Sheet 'Tabla de Calificación Bots' cargado exitosamente.
Primeras 5 filas de la tabla de calificación:


,Indicador,Ponderación,Excelente,Aceptable,Malo
0,Profit Factor,20,>2.0,1.5–2.0,<1.5
1,Win Rate,10,>65%,55–65%,<55%
2,Recovery Factor,15,>3,2–3,<2
3,Sharpe Ratio,15,>1.5,1–1.5,<1
4,Drawdown Máximo,20,<10%,10–20%,>20%


Columnas y tipos de datos de la tabla de calificación:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Indicador    7 non-null      object
 1   Ponderación  7 non-null      int64 
 2   Excelente    7 non-null      object
 3   Aceptable    7 non-null      object
 4   Malo         7 non-null      object
dtypes: int64(1), object(4)
memory usage: 412.0+ bytes


None

### Celda 14 - Definición de Criterios de Calificación de Robots

Ahora que la tabla de calificación (`rating_table_df`) se ha cargado correctamente, el siguiente paso es interpretar sus columnas para establecer los criterios de evaluación. Esta tabla define cómo se calificará a cada robot basándose en indicadores específicos:

*   **`Indicador`**: El nombre del indicador de rendimiento que se va a evaluar (por ejemplo, 'Profit Factor', 'Win Rate').
*   **`Ponderación`**: El peso o importancia de cada indicador en la calificación final.
*   **`Excelente`**, **`Aceptable`**, **`Malo`**: Los rangos o condiciones para calificar un indicador como Excelente, Aceptable o Malo. Estos valores necesitan ser parseados para ser utilizados en la lógica de calificación.

En la siguiente celda, procesaremos esta tabla para crear un conjunto de reglas programáticas que permitan calificar cada robot en función de sus métricas de rendimiento.

In [147]:
# Celda 15 - Procesar la Tabla de Calificación para definir los criterios

# La tabla `rating_table_df` ya está cargada y disponible.
# Vamos a transformar los criterios 'Excelente', 'Aceptable', 'Malo' en un formato utilizable.

import re # Asegurarse de que re esté importado para el parsing de regex

def parse_condition(condition_str, indicator_name):
    condition_str = str(condition_str).strip()

    # Manejo de casos específicos para 'Expectancy' (Positiva y alta, Positiva, Negativa)
    # Retornamos diccionarios para indicar explícitamente el tipo 'category'
    if indicator_name == 'Expectancy':
        if 'Positiva y alta' in condition_str: return {'type': 'category', 'value': 'Excelente'}
        if 'Positiva' in condition_str: return {'type': 'category', 'value': 'Aceptable'}
        if 'Negativa' in condition_str: return {'type': 'category', 'value': 'Malo'}
        return None

    # Primero, un manejo específico para 'Consistencia mensual' que contiene texto
    if indicator_name == 'Consistencia mensual' and 'meses positivos' in condition_str:
        # Extraer el número que precede a 'meses positivos', incluyendo posibles operadores como > o <
        match = re.search(r'([<>]?(\d+\.?\d*))%?\s*meses positivos', condition_str)
        if match:
            # Usar la parte numérica como la nueva condition_str_cleaned para el parsing genérico
            condition_str_cleaned = match.group(1).strip() # Ej: '>80'
        else:
            return None # No se pudo parsear el formato específico
    else:
        # Para otros indicadores y las condiciones restantes de 'Consistencia mensual' (60-80%, <60%)
        # eliminamos el '%' y luego aplicamos el parsing genérico.
        condition_str_cleaned = condition_str.replace('%', '').strip()

    # Convertir a min/max si es un rango (ej. 1.5–2.0, 55–65)
    if '–' in condition_str_cleaned:
        try:
            min_val, max_val = map(float, condition_str_cleaned.split('–'))
            # Devolver una tupla con valores numéricos flotantes
            return (min_val, max_val)
        except ValueError:
            pass # Fallback a otros parsers si falla

    # Mayor que (ej. >2.0, >65) o menor que (ej. <1.5, <10)
    if condition_str_cleaned.startswith('>'):
        try:
            # Devolver una tupla (min_val, +inf)
            return (float(condition_str_cleaned[1:]), float('inf'))
        except ValueError:
            pass
    elif condition_str_cleaned.startswith('<'):
        try:
            # Devolver una tupla (-inf, float(condition_str_cleaned[1:]))
            return (float('-inf'), float(condition_str_cleaned[1:]))
        except ValueError:
            pass

    # Si no se ajusta a ningún patrón conocido, devolver None
    return None


rating_criteria = {}
for index, row in rating_table_df.iterrows():
    indicator = row['Indicador']
    # Asegurarse de que el peso se lea como un número flotante para cálculos futuros
    # Eliminar '%' si existe y convertir a float
    weight = float(str(row['Ponderación']).replace('%', ''))

    excellent_condition = parse_condition(row['Excelente'], indicator)
    acceptable_condition = parse_condition(row['Aceptable'], indicator)
    bad_condition = parse_condition(row['Malo'], indicator)

    rating_criteria[indicator] = {
        'weight': weight,
        'Excelente': excellent_condition,
        'Aceptable': acceptable_condition,
        'Malo': bad_condition
    }

print("Criterios de calificación de robots procesados:")
for indicator, criteria in rating_criteria.items():
    print(f"- {indicator}:")
    print(f"  Ponderación: {criteria['weight']}%")
    print(f"  Excelente: {criteria['Excelente']}")
    print(f"  Aceptable: {criteria['Aceptable']}")
    print(f"  Malo: {criteria['Malo']}")

Criterios de calificación de robots procesados:
- Profit Factor:
  Ponderación: 20.0%
  Excelente: (2.0, inf)
  Aceptable: (1.5, 2.0)
  Malo: (-inf, 1.5)
- Win Rate:
  Ponderación: 10.0%
  Excelente: (65.0, inf)
  Aceptable: (55.0, 65.0)
  Malo: (-inf, 55.0)
- Recovery Factor:
  Ponderación: 15.0%
  Excelente: (3.0, inf)
  Aceptable: (2.0, 3.0)
  Malo: (-inf, 2.0)
- Sharpe Ratio:
  Ponderación: 15.0%
  Excelente: (1.5, inf)
  Aceptable: (1.0, 1.5)
  Malo: (-inf, 1.0)
- Drawdown Máximo:
  Ponderación: 20.0%
  Excelente: (-inf, 10.0)
  Aceptable: (10.0, 20.0)
  Malo: (20.0, inf)
- Expectancy:
  Ponderación: 10.0%
  Excelente: {'type': 'category', 'value': 'Excelente'}
  Aceptable: {'type': 'category', 'value': 'Aceptable'}
  Malo: {'type': 'category', 'value': 'Malo'}
- Consistencia mensual:
  Ponderación: 10.0%
  Excelente: (80.0, inf)
  Aceptable: (60.0, 80.0)
  Malo: (-inf, 60.0)


### Celda 16 - Implementación y Aplicación de la Lógica de Calificación de Robots

Con los criterios de calificación definidos, podemos ahora implementar una función que calculará la 'Calificación' para cada robot en tu `updated_master_df`. Esta función iterará a través de los indicadores disponibles para cada robot, comparará sus valores con los rangos 'Excelente', 'Aceptable' y 'Malo' definidos, y asignará una puntuación ponderada. Finalmente, convertirá esta puntuación en una calificación cualitativa (Ej. 'Excelente', 'Aceptable', 'Malo').

**Nota importante sobre indicadores no integrados completamente:**
Actualmente, los indicadores 'Sharpe Ratio', 'Expectancy' y 'Consistencia mensual' de la tabla de calificación no se han integrado completamente en `updated_master_df` (o requieren un procesamiento adicional para `Consistencia mensual` más allá del número total de meses). Por lo tanto, no se incluirán en el cálculo de la calificación en este momento. La calificación actual se basará en los indicadores disponibles: 'Profit Factor', 'Win Rate', 'Recovery Factor' y 'Drawdown Máximo'.

In [148]:
# Celda 17 - Función para calcular la calificación del robot y aplicación a la tabla maestra

def calculate_robot_rating(row, criteria):
    total_weighted_score = 0
    total_applicable_weight = 0

    # Mapeo de nombres de indicadores en rating_criteria a nombres de columna en updated_master_df
    indicator_to_column = {
        'Profit Factor': 'PF',
        'Win Rate': 'Probability', # Actualizado para usar 'Probability' en lugar de 'WR'
        'Recovery Factor': 'Recovery Factor',
        'Sharpe Ratio': 'Sharpe Ratio', # No está directamente en updated_master_df actual
        'Drawdown Máximo': 'Max drawdown', # Updated to 'Max drawdown'
        'Expectancy': 'Expectancy',   # No está directamente en updated_master_df actual
        'Consistencia mensual': 'Consistencia mensual' # Requiere cálculo de % meses positivos
    }

    for indicator_name, detail in criteria.items():
        column_name = indicator_to_column.get(indicator_name)

        # Si el indicador no tiene una columna mapeada en el df principal o si el indicador no está en el df
        if column_name is None or column_name not in row.index:
            # print(f"Advertencia: Indicador '{indicator_name}' no mapeado o no encontrado en el DataFrame. Saltando.")
            continue

        robot_value = row.get(column_name)

        if pd.isna(robot_value) or robot_value is None:
            # print(f"Advertencia: El valor del indicador '{indicator_name}' es nulo o no está disponible para el robot. Saltando este indicador.")
            continue # No incluir este indicador si su valor falta

        weight = detail['weight']

        score = 0
        if indicator_name == 'Expectancy':
            # Manejo de 'Expectancy' categórico (si estuviera disponible en el df)
            if isinstance(detail['Excelente'], dict) and robot_value == detail['Excelente']['value']:
                score = 100
            elif isinstance(detail['Aceptable'], dict) and robot_value == detail['Aceptable']['value']:
                score = 50
            elif isinstance(detail['Malo'], dict) and robot_value == detail['Malo']['value']:
                score = 0
            else:
                score = 0 # Si la categoría no coincide
        else:
            # Manejo de rangos numéricos
            # Valor para comparación: absoluto para Drawdown Máximo
            value_for_comparison = abs(robot_value) if indicator_name == 'Drawdown Máximo' else robot_value

            excel_cond = detail['Excelente']
            accept_cond = detail['Aceptable']
            bad_cond = detail['Malo']

            # Verificar Excelente
            if excel_cond and excel_cond[0] <= value_for_comparison <= excel_cond[1]:
                score = 100
            # Verificar Aceptable
            elif accept_cond and accept_cond[0] <= value_for_comparison <= accept_cond[1]:
                score = 50
            # Verificar Malo
            elif bad_cond and bad_cond[0] <= value_for_comparison <= bad_cond[1]:
                score = 0
            else:
                score = 0 # Valor fuera de los rangos definidos

        total_weighted_score += score * weight
        total_applicable_weight += weight

    if total_applicable_weight == 0:
        return pd.NA # No se pudieron aplicar indicadores o todos los valores estaban ausentes

    final_numeric_score = (total_weighted_score / total_applicable_weight)

    # Convertir puntuación numérica a calificación cualitativa
    if final_numeric_score >= 80:
        return 'Excelente'
    elif final_numeric_score >= 50:
        return 'Aceptable'
    else:
        return 'Malo'

# Aplicar la función de calificación a cada fila de updated_master_df
# La columna 'Calificación' ya fue inicializada en Celda 9 con pd.NA
updated_master_df['Calificación'] = updated_master_df.apply(lambda row: calculate_robot_rating(row, rating_criteria), axis=1)


print("Tabla Maestra Actualizada con Calificaciones (primeras 5 filas):")
display(updated_master_df.head())

# Guardar la tabla maestra actualizada con la calificación
# (Se replica la lógica de guardado de Celda 9 para asegurar que la versión final se persista)
miami_timezone = pytz.timezone('America/New_York')
current_date_str = datetime.now(miami_timezone).strftime('%Y-%m-%d_%H-%M-%S')
MASTER_TABLE_PATH_DATED = os.path.join(BASE_MASTER_TABLE_DIR, f"{BASE_MASTER_TABLE_NAME}_{current_date_str}.csv")

# Eliminar archivos de tabla maestra anteriores antes de guardar el nuevo
print("\nEliminando versiones anteriores de la tabla maestra antes de guardar la actualizada...")
for f_name in os.listdir(BASE_MASTER_TABLE_DIR):
    if f_name.startswith(BASE_MASTER_TABLE_NAME) and f_name.endswith('.csv'):
        file_path_to_delete = os.path.join(BASE_MASTER_TABLE_DIR, f_name)
        try:
            os.remove(file_path_to_delete)
            print(f"  - Eliminado: {f_name}")
        except Exception as e:
            print(f"  - Error al eliminar {f_name}: {e}")

updated_master_df.to_csv(MASTER_TABLE_PATH_DATED, index=False)
print(f"\nTabla maestra actualizada (con calificaciones) guardada en: {MASTER_TABLE_PATH_DATED}")

Tabla Maestra Actualizada con Calificaciones (primeras 5 filas):


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,Total # of trades,Total net profit,Net Profit/Mes,PF,Probability,Max drawdown,Recovery Factor,PayoffRatio,Avg Win,Avg Loss,Calificación,Comentarios
0,RB001,T001,2026-01-01 00:00:00,2026-06-30 00:00:00,MNQ,5.913272,550.0,5680.5,960.635667,1.23,12.52,-6111.8,0.929432,7.232887,378.28,-52.3,Malo,



Eliminando versiones anteriores de la tabla maestra antes de guardar la actualizada...
  - Eliminado: Tabla Maestra Bots_2026-08-20_16-31-52.csv

Tabla maestra actualizada (con calificaciones) guardada en: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots_2026-08-20_16-32-39.csv


### Próximos pasos: Refinamiento de la Calificación y Adición de Indicadores

Para mejorar aún más la precisión de la calificación, los próximos pasos podrían incluir:

1.  **Adición de 'Sharpe Ratio' y 'Expectancy':** Estos indicadores deberían ser extraídos del análisis de estrategia y añadidos a tu `updated_master_df` para que puedan ser incluidos en el cálculo de la calificación.

2.  **Cálculo de 'Consistencia mensual':** Este indicador requiere un procesamiento más detallado de los datos brutos para determinar el porcentaje de meses positivos. Esto podría implementarse en una celda de procesamiento de datos para luego integrarlo en el `updated_master_df`.

---

### Celda 18 - Revisión de Indicadores Calculados

A continuación, se muestra la tabla maestra `updated_master_df` con todos los indicadores que se han calculado y la `Calificación` final. Por favor, revisa cada columna y dime tus impresiones. Podemos ajustar cualquier cálculo o enfoque que consideres necesario.

In [149]:
# Mostrar el DataFrame actualizado con todos los indicadores
print("Tabla Maestra Actualizada con todos los Indicadores y Calificaciones:")
display(updated_master_df)

Tabla Maestra Actualizada con todos los Indicadores y Calificaciones:


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,Total # of trades,Total net profit,Net Profit/Mes,PF,Probability,Max drawdown,Recovery Factor,PayoffRatio,Avg Win,Avg Loss,Calificación,Comentarios
0,RB001,T001,2026-01-01 00:00:00,2026-06-30 00:00:00,MNQ,5.913272,550.0,5680.5,960.635667,1.23,12.52,-6111.8,0.929432,7.232887,378.28,-52.3,Malo,


In [150]:
print('Revisemos los indicadores de la tabla maestra:')
display(updated_master_df)

Revisemos los indicadores de la tabla maestra:


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,Total # of trades,Total net profit,Net Profit/Mes,PF,Probability,Max drawdown,Recovery Factor,PayoffRatio,Avg Win,Avg Loss,Calificación,Comentarios
0,RB001,T001,2026-01-01 00:00:00,2026-06-30 00:00:00,MNQ,5.913272,550.0,5680.5,960.635667,1.23,12.52,-6111.8,0.929432,7.232887,378.28,-52.3,Malo,


### Fórmulas de Indicadores

*   **Recovery Factor**: Se calcula como `Total net profit` / `abs(Max drawdown)`.
    *   **Explicación**: Mide la eficiencia con la que un sistema recupera las pérdidas en relación con el drawdown máximo. Un valor más alto indica una mejor capacidad de recuperación.

In [151]:
print('Revisando la columna Calificación en updated_master_df:')
display(updated_master_df[['Robot', 'Numero del Test', 'Instrumento', 'PF', 'Probability', 'Recovery Factor', 'Max drawdown', 'Calificación', 'Comentarios']])

Revisando la columna Calificación en updated_master_df:


,Robot,Numero del Test,Instrumento,PF,Probability,Recovery Factor,Max drawdown,Calificación,Comentarios
0,RB001,T001,MNQ,1.23,12.52,0.929432,-6111.8,Malo,


### Revisión de Columnas de la Tabla Maestra

A continuación, se listan las columnas de la `updated_master_df` con su posible origen. Por favor, confirma si esto es correcto o si necesitas realizar alguna modificación en la descripción o el origen de cada columna.

- **Robot**: Identificador único del robot. Origen: **Calculado** (extraído del nombre del archivo).
- **Numero del Test**: Identificador del test del robot. Origen: **Calculado** (extraído del nombre del archivo).
- **Fecha Inicio**: Fecha de inicio del backtest. Origen: **NinjaTrader** (extraído del archivo crudo).
- **Fecha Fin**: Fecha de fin del backtest. Origen: **NinjaTrader** (extraído del archivo crudo).
- **Instrumento**: Instrumento financiero utilizado. Origen: **Calculado** (extraído del nombre del archivo).
- **# Meses**: Número de meses cubiertos por el backtest. Origen: **Calculado** (diferencia entre Fecha Fin y Fecha Inicio).
- **Total # of trades**: Número total de operaciones. Origen: **NinjaTrader**.
- **Total net profit**: Ganancia neta total. Origen: **NinjaTrader**.
- **Net Profit/Mes**: Ganancia neta promedio por mes. Origen: **Calculado** (`Total net profit` / `# Meses`).
- **PF (Profit Factor)**: Factor de Beneficio. Origen: **NinjaTrader**.
- **Probability**: Porcentaje de operaciones ganadoras. Origen: **NinjaTrader** (Probability).
- **Max drawdown**: Pérdida máxima desde un pico hasta un valle. Origen: **NinjaTrader**.
- **Recovery Factor**: Factor de Recuperación. Origen: **Calculado** (`Total net profit` / `abs(Max drawdown)`).
- **PayoffRatio**: Ratio de Ganancia Media sobre Pérdida Media. Origen: **Calculado** (`Avg Win` / `abs(Avg Loss)`).
- **Avg Win**: Ganancia promedio por operación ganadora. Origen: **NinjaTrader**.
- **Avg Loss**: Pérdida promedio por operación perdedora. Origen: **NinjaTrader**.
- **Calificación**: Calificación cualitativa del robot. Origen: **Calculado** (basado en `rating_criteria`).
- **Comentarios**: Comentarios manuales sobre el robot/test. Origen: **Usuario**.